Task 4 – Stacked Area Chart Analysis

## Objective

To visualize cumulative installs over time for selected app categories using a stacked area chart.

## Dataset

Google Play Store Dataset

## Transformations Applied

* Rating ≥ 4.2
* Reviews > 1,000
* App Size between 20 MB and 80 MB
* Categories starting with T or P
* Excluded app names containing numeric values
* Applied multilingual category translations

## KPI Measured

* Cumulative Installs
* Month-over-Month Install Growth

## Visualization

Stacked Area Chart with category-wise install contribution and highlighted growth periods exceeding 25%.

## Time Restriction

The chart is available only between 4 PM IST and 6 PM IST.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [2]:
apps_df = pd.read_csv(
    r"C:\Users\HP\Downloads\archive\googleplaystore.csv"
)

apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [3]:
apps_df = apps_df[
    apps_df['Size'] != 'Varies with device'
]

apps_df['Size'] = (
    apps_df['Size']
    .str.replace('M','')
    .str.replace('k','')
)

apps_df['Size'] = pd.to_numeric(
    apps_df['Size'],
    errors='coerce'
)

apps_df['Reviews'] = pd.to_numeric(
    apps_df['Reviews'],
    errors='coerce'
)

apps_df['Last Updated'] = pd.to_datetime(
    apps_df['Last Updated'],
    errors='coerce'
)

In [4]:
apps_df['Month'] = (
    apps_df['Last Updated']
    .dt.to_period('M')
    .astype(str)
)

In [5]:
apps_df = apps_df[
    ~apps_df['App']
    .str.contains(r'\d', regex=True, na=False)
]

In [6]:
filtered_df = apps_df[
    (apps_df['Rating'] >= 4.2) &
    (apps_df['Reviews'] > 1000) &
    (apps_df['Size'] >= 20) &
    (apps_df['Size'] <= 80) &
    (
        apps_df['Category'].str.startswith('T') |
        apps_df['Category'].str.startswith('P')
    )
]

filtered_df.shape

(142, 14)

In [7]:
filtered_df['Category'].value_counts()

Category
PHOTOGRAPHY         55
TRAVEL_AND_LOCAL    32
PRODUCTIVITY        22
TOOLS               18
PERSONALIZATION     11
PARENTING            4
Name: count, dtype: int64

In [12]:
category_translation = {
    'TRAVEL_AND_LOCAL': 'Voyage et Local',   # French
    'PRODUCTIVITY': 'Productividad',         # Spanish
    'PHOTOGRAPHY': 'Photography in Japanese'                    # Japanese
}

filtered_df['Category_Display'] = (
    filtered_df['Category']
    .replace(category_translation)
)

In [9]:
filtered_df['Installs'] = (
    filtered_df['Installs']
    .str.replace(',', '')
    .str.replace('+', '')
)

filtered_df['Installs'] = pd.to_numeric(
    filtered_df['Installs'],
    errors='coerce'
)

In [10]:
area_df = (
    filtered_df
    .groupby(['Month', 'Category_Display'])['Installs']
    .sum()
    .unstack(fill_value=0)
)

area_df.head()

Category_Display,PARENTING,PERSONALIZATION,Productividad,TOOLS,Voyage et Local,写真
Month,,,,,,
2014-01,0,0,500000,0,0,0
2014-11,0,0,0,0,0,1000000
2015-08,0,0,0,0,0,10000
2016-10,0,0,1000000,0,0,0
2016-12,0,2000000,0,0,0,0


In [14]:
growth = area_df.sum(axis=1).pct_change()

growth_months = growth[growth > 0.25]

growth_months

Month
2014-11     1.000000
2016-10    99.000000
2016-12     1.000000
2017-03    24.000000
2017-07     0.500000
2017-09    19.000000
2017-10     4.050000
2017-12     0.980198
2018-01     1.100000
2018-04     1.380952
2018-05     5.200000
2018-06     2.774194
2018-07    10.656410
2018-08     0.916410
dtype: float64

In [16]:
category_translation = {
    'TRAVEL_AND_LOCAL': 'Voyage et Local',
    'PRODUCTIVITY': 'Productividad',
    'PHOTOGRAPHY': 'Shashin'
}

In [17]:
filtered_df['Category_Display'] = (
    filtered_df['Category']
    .replace(category_translation)
)

In [18]:
area_df = (
    filtered_df
    .groupby(['Month', 'Category_Display'])['Installs']
    .sum()
    .unstack(fill_value=0)
)

In [19]:
growth = area_df.sum(axis=1).pct_change()

growth_months = growth[growth > 0.25].index

In [21]:
from datetime import datetime
import pytz
import matplotlib.pyplot as plt

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist)

if 16 <= current_time.hour < 18:

    growth = area_df.sum(axis=1).pct_change()

    growth_months = growth[growth > 0.25].index

    fig, ax = plt.subplots(figsize=(14,7))

    area_df.plot.area(
        ax=ax,
        stacked=True,
        alpha=0.8
    )

    # Highlight growth months
    for month in growth_months:

        ax.axvspan(
            area_df.index.get_loc(month)-0.5,
            area_df.index.get_loc(month)+0.5,
            alpha=0.15
        )

    plt.title(
        "Cumulative Installs Over Time by Category"
    )

    plt.xlabel("Month")
    plt.ylabel("Installs")

    plt.legend(
        title="Category",
        bbox_to_anchor=(1.02,1),
        loc='upper left'
    )

    plt.tight_layout()
    plt.show()

else:
    print(
        "Stacked area chart is available only between 4 PM IST and 6 PM IST."
    )

Stacked area chart is available only between 4 PM IST and 6 PM IST.
